# Movie Review Sentiment Analysis

This notebook performs sentiment analysis on the IMDB Movie Review dataset. The project involves data collection, exploratory data analysis (EDA), text preprocessing, feature extraction using TF-IDF, and sentiment classification using various machine learning models including Logistic Regression, Random Forest, and a Neural Network with Sentence Embeddings. The goal is to predict whether a movie review is positive or negative.

# Dataset Collection and Loading

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [ ]:
!mkdir Dataset
!unzip imdb-dataset-of-50k-movie-reviews.zip -d Dataset

# Reading The Data

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Dataset/IMDB Dataset.csv')

print(df)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicates:", df.duplicated().sum())

In [ ]:
print("Shape Before:", df.shape)
df = df.drop_duplicates()
print("Shape After:", df.shape)

In [ ]:
print("Duplicates:", df.duplicated().sum())

In [ ]:
df["sentiment"].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='sentiment', data=df)
plt.show()

# Text EDA

In [ ]:
df['wordCount'] = df['review'].apply(lambda x: len(str(x).split()))

In [ ]:
df['charCount'] = df['review'].apply(lambda x: len(str(x)))

In [ ]:
df

In [ ]:
print(df.describe())

**Do people write longer reviews when they dislike a movie?**                                                                                                             

**People Generally Write the Larger review then the Sentiment is Positive**

In [ ]:
sns.boxplot(x='sentiment' , y='wordCount',data=df)

**counts the number of reviews containing HTML tags.**
**`na=False` treats missing (NaN) values as `False`**
**while checking the condition, preventing errors and ensuring consistent results.**


In [ ]:
df['review'].str.contains('<', na=False).sum()

In [ ]:
from bs4 import BeautifulSoup

def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()

df['review'] = df['review'].apply(remove_html_tags)
df

In [ ]:
# df['sentiment'] = df['sentiment'].map({'positive': 1,'negative': 0})
# Map 'positive' to 1 and 'negative' to 0 for the sentiment column
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [ ]:
df.head()

In [ ]:
df['review'] = df['review'].str.lower()

**check of data contain any urls**

In [ ]:
df['review'].str.contains('http', na=False).sum()

**Removed**

In [ ]:
import re

df['review'] = df['review'].apply(
    lambda x: re.sub(r'http\S+|www\S+', '', x)
)

In [ ]:
df['review'].str.contains('http', na=False).sum()

**Removed special characters, numbers, and punctuation from the review text while retaining only alphabetic characters. Additionally, multiple spaces were replaced with a single space and leading/trailing spaces were removed to create clean and consistent text for further NLP preprocessing**.


In [ ]:
import re

df['review'] = df['review'].apply(
    lambda x: re.sub(r'\s+', ' ', re.sub(r'[^a-zA-Z\s]', '', x)).strip()
)

**Removing Stopword**

**Removed common words such as the, is, was, and and that occur frequently but contribute little to sentiment prediction. This helps reduce noise and allows the model to focus on more meaningful words.**

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stopWords = set(stopwords.words('english')) #collecting all the stop words in nlkt library

print(stopWords)
print("Total Stop words in NLKT library:",len(stopWords))

In [ ]:
def removeStopWords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stopWords]
    return ' '.join(filtered_words)

# Apply the clean function to your column
df['review'] = df['review'].apply(removeStopWords)

**Lemmatization**
Converted words to their base form, such as *movies → movie* and *liked → like*. This reduces vocabulary size and helps the model treat different forms of the same word as a single feature. **bold text**


In [ ]:
import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def lemmatizetext(text):

    words = text.split()
    lemmatizedWords = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatizedWords)


df['review'] = df['review'].apply(lemmatizetext)

**Printing the most frequent Word in reviews**

**Analyzed the most frequently occurring words in the cleaned reviews to identify dominant terms present in the dataset. This helps understand the overall vocabulary and common themes discussed in movie reviews.**

In [ ]:
from collections import Counter

allWords = ' '.join(df['review']).split()
wordFreq = Counter(allWords)

for word, freq in wordFreq.most_common(20):
    print(f"{word}: {freq}")

In [ ]:
positiveReviews = ' '.join(df[df['sentiment'] == 1]['review']).split()

for word,freq in Counter(positiveReviews).most_common(20):
    print(f"{word}: {freq}")

In [ ]:
# Also recalculate negativeReviews for completeness
negativeReviews = ' '.join(df[df['sentiment'] == 0]['review']).split()

for word, freq in Counter(negativeReviews).most_common(20):
    print(f"{word}: {freq}")

# Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

In [ ]:
x = tfidf.fit_transform(df['review'])
print("TF-IDF Matrix Shape:", x.shape)

**Extract Target Variable**

In [ ]:
y = df['sentiment']
print(y.head())

**Check Vocabulary Size**

In [ ]:
print("Vocabulary Size:", len(tfidf.vocabulary_))

**Display First 20 Features**

In [ ]:
featureNames = tfidf.get_feature_names_out()
print(featureNames[:20])

**View TF-IDF Matrix for First 5 Reviews**

In [ ]:
tfidfDf = pd.DataFrame(
    x[:5].toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidfDf.head()

In [ ]:
tfidfScores = x.sum(axis=0).A1

featureImportance = pd.DataFrame({
    'feature': tfidf.get_feature_names_out(),
    'score': tfidfScores
})

featureImportance = featureImportance.sort_values(
    by='score',
    ascending=False
)

featureImportance.head(20)

In [ ]:
import matplotlib.pyplot as plt

topFeatures = featureImportance.head(20)

plt.figure(figsize=(10, 6))
plt.barh(
    topFeatures['feature'],
    topFeatures['score']
)

plt.xlabel("TF-IDF Score")
plt.ylabel("Features")
plt.title("Top 20 TF-IDF Features")
plt.gca().invert_yaxis()

plt.show()

**Train Test Validation Spliting(70 + 30)**



In [ ]:
from sklearn.model_selection import train_test_split

xTrain, xTemp, yTrain, yTemp = train_test_split(x,y,test_size=0.30,random_state=42,stratify=y)
xValidation, xTest, yValidation, yTest = train_test_split(xTemp,yTemp,test_size=0.50,random_state=42,stratify=yTemp)

In [ ]:
print("Training Set Shape   :", xTrain.shape)
print("Validation Set Shape :", xValidation.shape)
print("Testing Set Shape    :", xTest.shape)

**Cheaking CLass distribution**

In [ ]:
print("Training Set:" , yTrain.value_counts())
print("Validation Set:" , yValidation.value_counts())
print("Testing Set:" , yTest.value_counts())

**Checking Sparsity in the TF-IDF Matrix**

In [ ]:
print("Number of Reviews :", x.shape[0])
print("Number of Features:", x.shape[1])
print("Non-Zero Elements:", x.nnz)

# Using ML Models

### Logistic Regression

**Training the Model (Logistic regression)**

This code block initializes a Logistic Regression model from scikit-learn.  
- `from sklearn.linear_model import LogisticRegression`: Imports the `LogisticRegression` class.  
- `logisticRegressionModel = LogisticRegression(max_iter=1000, random_state=42)`: Creates an instance of the model.  
  - `max_iter=1000`: Sets the maximum number of iterations for the solver to converge.  
  - `random_state=42`: Sets the seed for reproducibility of random operations.  
- The last line `logisticRegressionModel` simply displays the model object, showing its current parameters.

In [ ]:
from sklearn.linear_model import LogisticRegression
logisticRegressionModel = LogisticRegression (max_iter=1000,random_state=42)

logisticRegressionModel

This line trains the `logisticRegressionModel` using the training data.  
- `logisticRegressionModel.fit(xTrain, yTrain)`: Fits the model to the training features (`xTrain`) and training labels (`yTrain`). The model learns the relationship between the features and the sentiment from this data.

In [ ]:
logisticRegressionModel.fit(xTrain,yTrain)

This code block makes predictions on the validation set.  
- `yValidationPredictions = logisticRegressionModel.predict(xValidation)`: Uses the trained model to predict sentiment labels for the validation features (`xValidation`).  
- `yValidationPredictions[:10]`: Displays the first 10 predicted labels from the validation set, allowing for a quick check of the predictions.

In [ ]:
yValidationPredictions = logisticRegressionModel.predict(xValidation)
yValidationPredictions[:10]

This code calculates and prints the accuracy of the model on the validation set.  
- `from sklearn.metrics import accuracy_score`: Imports the `accuracy_score` function to compare actual and predicted labels.  
- `validationAccuracy = accuracy_score(yValidation, yValidationPredictions)`: Computes the accuracy by comparing the true validation labels (`yValidation`) with the model's predictions (`yValidationPredictions`).  
- `print("Validation Accuracy:", validationAccuracy)`: Prints the calculated accuracy.

In [ ]:
from sklearn.metrics import accuracy_score
validationAccuracy = accuracy_score(yValidation,yValidationPredictions)
print("Validation Accuracy:",validationAccuracy)

This code block generates and prints a detailed classification report for the validation set.  
- `from sklearn.metrics import classification_report`: Imports the `classification_report` function.  
- `print(classification_report(yValidation, yValidationPredictions))`: Prints a report that includes precision, recall, f1-score, and support for each class, providing a comprehensive evaluation of the model's performance on the validation data.

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(yValidation,yValidationPredictions))

This code calculates and prints the confusion matrix for the validation set.  
- `from sklearn.metrics import confusion_matrix`: Imports the `confusion_matrix` function.  
- `confusionMatrix = confusion_matrix(yValidation, yValidationPredictions)`: Computes the confusion matrix, which shows the number of correct and incorrect predictions made by the classification model when compared against the actual outcomes (`yValidation` vs `yValidationPredictions`).  
- `print(confusionMatrix)`: Displays the confusion matrix in a raw array format.

In [ ]:
from sklearn.metrics import confusion_matrix
confusionMatrix = confusion_matrix(yValidation,yValidationPredictions)
print(confusionMatrix)

This code visualizes the confusion matrix for the validation set using a heatmap.  
- `import seaborn as sns`: Imports the Seaborn library for statistical data visualization.  
- `import matplotlib.pyplot as plt`: Imports Matplotlib for plotting.  
- `plt.figure(figsize=(6, 5))`: Creates a figure with a specified size for the plot.  
- `sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Blues')`: Generates a heatmap of the `confusionMatrix`.  
  - `annot=True`: Annotates the heatmap cells with the data values.  
  - `fmt='d'`: Formats the annotations as integers.  
  - `cmap='Blues'`: Uses a blue color map for the heatmap.  
- `plt.xlabel("Predicted Label")`, `plt.ylabel("Actual Label")`, `plt.title("Logistic Regression Confusion Matrix")`: Sets the labels for the x-axis, y-axis, and the title of the plot.  
- `plt.show()`: Displays the generated plot.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))

sns.heatmap(confusionMatrix,annot=True,fmt='d',cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Logistic Regression Confusion Matrix")

plt.show()

First, we need to generate predictions for the test set.  
- `yTestPredictions = logisticRegressionModel.predict(xTest)`: Uses the trained model to predict sentiment labels for the test features (`xTest`).

In [ ]:
yTestPredictions = logisticRegressionModel.predict(xTest)

In [ ]:
from sklearn.metrics import accuracy_score
testAccuracy = accuracy_score(yTest, yTestPredictions)
print("Test Accuracy:", testAccuracy)

This code visualizes the confusion matrix for the test set using a heatmap.  
- `testConfusionMatrix = confusion_matrix(yTest, yTestPredictions)`: Computes the confusion matrix for the test set by comparing actual test labels (`yTest`) with the model's predictions (`yTestPredictions`).  
- The subsequent lines are similar to the validation confusion matrix visualization, but applied to the test set for an unbiased evaluation of the final model performance.

In [ ]:
testConfusionMatrix = confusion_matrix(yTest,yTestPredictions)

plt.figure(figsize=(6, 5))

sns.heatmap(testConfusionMatrix,annot=True,fmt='d',cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Logistic Regression Test Confusion Matrix")

plt.show()

### Random Forest

This line imports the `RandomForestClassifier` class from scikit-learn's `ensemble` module, which is used to build the Random Forest model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

This code block initializes a Random Forest Classifier model.  
- `randomForestModel = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)`: Creates an instance of the model.  
  - `n_estimators=100`: Specifies the number of trees in the forest (100 in this case). More trees generally lead to better performance but also longer training times.  
  - `random_state=42`: Sets the seed for reproducibility of random operations, ensuring the same results each time the code is run.  
  - `n_jobs=-1`: Uses all available CPU cores for parallel processing, speeding up training.  
- The last line `randomForestModel` simply displays the model object, showing its current parameters.

In [ ]:
randomForestModel = RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)

randomForestModel

This line trains the `randomForestModel` using the training data.  
- `randomForestModel.fit(xTrain, yTrain)`: Fits the model to the training features (`xTrain`) and training labels (`yTrain`), allowing the model to learn patterns for sentiment classification.

In [ ]:
randomForestModel.fit(xTrain,yTrain)

This code block makes predictions on the validation set using the trained Random Forest model.  
- `yValidationPredictionsRF = randomForestModel.predict(xValidation)`: Uses the trained `randomForestModel` to predict sentiment labels for the validation features (`xValidation`).  
- `yValidationPredictionsRF[:10]`: Displays the first 10 predicted labels from the validation set, providing a quick look at the model's output.

In [ ]:
yValidationPredictionsRF = randomForestModel.predict(xValidation)

yValidationPredictionsRF[:10]

This code calculates and prints the accuracy of the Random Forest model on the validation set.  
- `from sklearn.metrics import accuracy_score`: Imports the `accuracy_score` function for evaluating classification models.  
- `validationAccuracyRF = accuracy_score(yValidation, yValidationPredictionsRF)`: Compares the actual validation labels (`yValidation`) with the model's predictions (`yValidationPredictionsRF`) to compute the accuracy.  
- `print("Validation Accuracy:", validationAccuracyRF)`: Displays the calculated validation accuracy.

In [ ]:
from sklearn.metrics import accuracy_score

validationAccuracyRF = accuracy_score(yValidation, yValidationPredictionsRF)

print( "Validation Accuracy:", validationAccuracyRF)

This code block generates and prints a detailed classification report for the Random Forest model on the validation set.  
- `from sklearn.metrics import classification_report`: Imports the `classification_report` function.  
- `print(classification_report(yValidation, yValidationPredictionsRF))`: Provides a comprehensive report including precision, recall, f1-score, and support for each class, offering a deeper insight into the model's performance beyond just accuracy.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(yValidation,yValidationPredictionsRF))

This code calculates and prints the confusion matrix for the Random Forest model on the validation set.  
- `from sklearn.metrics import confusion_matrix`: Imports the `confusion_matrix` function.  
- `confusionMatrixRF = confusion_matrix(yValidation, yValidationPredictionsRF)`: Computes the confusion matrix, which summarizes the performance of the classification model by showing the counts of true positive, true negative, false positive, and false negative predictions.  
- `print(confusionMatrixRF)`: Displays the confusion matrix in a numerical array format.

In [ ]:
from sklearn.metrics import confusion_matrix

confusionMatrixRF = confusion_matrix(yValidation,yValidationPredictionsRF)

print(confusionMatrixRF)

This code visualizes the confusion matrix for the Random Forest model's validation set as a heatmap.  
- `import seaborn as sns` and `import matplotlib.pyplot as plt`: Imports the necessary libraries for plotting.  
- `plt.figure(figsize=(6, 5))`: Sets the size of the plot for better readability.  
- `sns.heatmap(confusionMatrixRF, annot=True, fmt='d', cmap='Blues')`: Creates the heatmap.  
  - `annot=True`: Displays the numerical values in each cell.  
  - `fmt='d'`: Formats the annotations as integers.  
  - `cmap='Blues'`: Uses a color scheme where darker blues indicate higher values.  
- `plt.xlabel("Predicted Label")`, `plt.ylabel("Actual Label")`, `plt.title("Random Forest Validation Confusion Matrix")`: Adds labels to the axes and a title to the plot for clarity.  
- `plt.show()`: Displays the generated heatmap.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

sns.heatmap(confusionMatrixRF,annot=True,fmt='d',cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Random Forest Validation Confusion Matrix")

plt.show()

This code block generates predictions for the test set using the trained Random Forest model.  
- `yTestPredictionsRF = randomForestModel.predict(xTest)`: Uses the `randomForestModel` to predict sentiment labels for the unseen test features (`xTest`). This is a crucial step for evaluating the model's generalization ability.  
- `yTestPredictionsRF[:10]`: Displays the first 10 predicted labels from the test set, allowing a quick check of the predictions.

In [ ]:
yTestPredictionsRF = randomForestModel.predict(xTest)

yTestPredictionsRF[:10]

This code calculates and prints the accuracy of the Random Forest model on the test set.  
- `testAccuracyRF = accuracy_score(yTest, yTestPredictionsRF)`: Compares the actual test labels (`yTest`) with the model's predictions (`yTestPredictionsRF`) to determine the model's accuracy on unseen data.  
- `print("Test Accuracy:", testAccuracyRF)`: Displays the final test accuracy.

In [ ]:
testAccuracyRF = accuracy_score(yTest,yTestPredictionsRF)

print("Test Accuracy:",testAccuracyRF)

This code block generates and prints a detailed classification report for the Random Forest model on the test set.  
- `print(classification_report(yTest, yTestPredictionsRF))`: Provides a comprehensive evaluation of the model's performance on the test data, including precision, recall, f1-score, and support for each class. This report is essential for understanding how well the model generalizes to new, unseen data.

In [ ]:
print(classification_report(yTest,yTestPredictionsRF))

This code calculates and prints the confusion matrix for the Random Forest model on the test set.  
- `testConfusionMatrixRF = confusion_matrix(yTest, yTestPredictionsRF)`: Computes the confusion matrix, comparing the true labels of the test set (`yTest`) with the predictions made by the model (`yTestPredictionsRF`). This matrix helps identify where the model is making correct and incorrect classifications on the test data.  
- `print(testConfusionMatrixRF)`: Displays the calculated confusion matrix in a numerical array format.

In [ ]:
testConfusionMatrixRF = confusion_matrix(yTest,yTestPredictionsRF)

print(testConfusionMatrixRF)

This code visualizes the confusion matrix for the Random Forest model's test set as a heatmap.  
- `plt.figure(figsize=(6, 5))`: Sets the size of the plot.  
- `sns.heatmap(testConfusionMatrixRF, annot=True, fmt='d', cmap='Blues')`: Creates the heatmap, similar to the validation confusion matrix visualization, but using the `testConfusionMatrixRF`.  
- `plt.xlabel("Predicted Label")`, `plt.ylabel("Actual Label")`, `plt.title("Random Forest Test Confusion Matrix")`: Adds descriptive labels and a title to the plot.  
- `plt.show()`: Displays the final heatmap of the test confusion matrix.

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(testConfusionMatrixRF, annot=True,fmt='d',cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Random Forest Test Confusion Matrix")

plt.show()

This code block creates a pandas DataFrame to compare the performance metrics (Validation Accuracy and Test Accuracy) of the Logistic Regression and Random Forest models.  
- `comparisonResults`: A dictionary holding the model names and their corresponding accuracy scores.  
- `import pandas as pd`: Imports the pandas library.  
- `comparisonDataFrame = pd.DataFrame(comparisonResults)`: Converts the dictionary into a DataFrame.  
- `comparisonDataFrame`: Displays the resulting DataFrame, which provides a clear side-by-side comparison of the two models' performance.

In [ ]:
comparisonResults = {
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Validation Accuracy": [
        validationAccuracy,
        validationAccuracyRF
    ],
    "Test Accuracy": [
        testAccuracy,
        testAccuracyRF
    ]
}

import pandas as pd

comparisonDataFrame = pd.DataFrame(
    comparisonResults
)

comparisonDataFrame

## Conclusion: Model Performance Comparison
After training and evaluating both Logistic Regression and Random Forest models for sentiment analysis on the IMDB movie review dataset, we can draw the following conclusions regarding their performance:

### 1. Overall Performance Metrics

| Model                 | Validation Accuracy | Test Accuracy |
|:----------------------|:--------------------|:--------------|
| Logistic Regression   | 0.884               | 0.887         |
| Random Forest         | 0.844               | 0.839         |

From the comparison table, it is evident that the **Logistic Regression model performed better** than the Random Forest model on this specific dataset.

### 2. Detailed Analysis:

**Logistic Regression:**
-   **Validation Accuracy (0.884):** The model showed strong performance on the validation set, indicating good generalization to unseen data during development.
-   **Test Accuracy (0.887):** The test accuracy is slightly higher than the validation accuracy, suggesting that the model maintained its performance well on completely new, unseen data. This indicates a robust and generalizable model.
-   **Classification Report:** The precision, recall, and f1-score for both positive and negative classes were consistently high (around 0.88-0.90), suggesting that the model is well-balanced in identifying both positive and negative sentiments without significant bias towards one class.
-   **Confusion Matrix:** The confusion matrix for Logistic Regression shows a relatively low number of false positives and false negatives, meaning it correctly classified a high percentage of both positive and negative reviews.

**Random Forest:**
-   **Validation Accuracy (0.844):** While respectable, the validation accuracy was noticeably lower than that of Logistic Regression.
-   **Test Accuracy (0.839):** The test accuracy was also lower than Logistic Regression and slightly decreased from its validation accuracy, indicating a slightly poorer generalization to new data compared to Logistic Regression.
-   **Classification Report:** The precision, recall, and f1-score for both classes were around 0.84-0.85, which is good but not as high as Logistic Regression. This suggests the model is slightly less effective at distinguishing between classes.
-   **Confusion Matrix:** The Random Forest confusion matrix revealed a higher number of misclassifications (false positives and false negatives) compared to Logistic Regression, particularly in distinguishing between the two sentiment classes.

### 3. Possible Reasons for Performance Differences:

-   **Model Complexity:** Logistic Regression is a linear model, while Random Forest is an ensemble of decision trees. For this particular text classification task with TF-IDF features, a simpler linear model like Logistic Regression might have captured the underlying patterns more effectively, especially if the relationship between features and sentiment is predominantly linear or linearly separable in the high-dimensional feature space created by TF-IDF.
-   **TF-IDF Effectiveness:** TF-IDF features, being high-dimensional and sparse, often work very well with linear models such as Logistic Regression and SVMs. These models can efficiently handle many features and find the linear decision boundary.
-   **Hyperparameter Tuning:** While both models were initialized with default parameters (`random_state=42`, `n_estimators=100` for Random Forest, `max_iter=1000` for Logistic Regression), further hyperparameter tuning might potentially improve the Random Forest model's performance. However, in this direct comparison, Logistic Regression showed superior performance out-of-the-box.

### 4. Final Recommendation:

Based on the evaluation, the **Logistic Regression model is the preferred choice** for this sentiment analysis task due to its higher validation and test accuracies, and overall better performance across various metrics. It demonstrates strong generalization capabilities and effectively classifies movie review sentiments.

# Neural Networks and Sentence Embeddings

In [ ]:
!pip install -U sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer
EmbeddingModel = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

In [ ]:
reviews = df['review'].tolist()
labels = df['sentiment'].values

In [ ]:
reviews = df['review'].tolist()

embeddings = EmbeddingModel.encode(
    reviews,
    batch_size=32,
    show_progress_bar=True
)

print(embeddings.shape)

In [ ]:
import numpy as np

# Features
x = np.array(embeddings)

# Labels
y = np.array(labels)

print("X Shape:", x.shape)
print("Y Shape:", y.shape)

print("X Data Type:", type(x))
print("Y Data Type:", type(y))

In [ ]:
from sklearn.model_selection import train_test_split

xTrain, xTemp, yTrain, yTemp = train_test_split(x,y,test_size=0.30,random_state=42,stratify=y)
xValidation, xTest, yValidation, yTest = train_test_split(xTemp,yTemp,test_size=0.50,random_state=42,stratify=yTemp)

In [ ]:
print("xTrain Shape:" , xTrain.shape)
print("xvalidation Shape:", xValidation.shape)
print("xTest.shape", xTest.shape)

In [ ]:
import keras

In [ ]:
import tensorflow as tf


# 1. Initialize an empty Sequential model
nnmodel = tf.keras.Sequential()

# 2. Add the layers one by one
nnmodel.add(tf.keras.layers.Dense(256, activation='relu', input_shape=(384,)))
nnmodel.add(tf.keras.layers.Dropout(0.3))
nnmodel.add(tf.keras.layers.BatchNormalization())

nnmodel.add(tf.keras.layers.Dense(128, activation='relu'))
nnmodel.add(tf.keras.layers.Dropout(0.3))
nnmodel.add(tf.keras.layers.BatchNormalization())


nnmodel.add(tf.keras.layers.Dense(1, activation='sigmoid'))

In [ ]:
nnmodel.summary()

In [ ]:
nnmodel.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### looking at above Diagram the epoch after 5 is usless so keeping epochs = 4 is quite good decision to reduce overfitting and get the best accuracy out there


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define EarlyStopping callback
earlyStopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=3,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored quantity
)

trainedModel = nnmodel.fit(
    xTrain,
    yTrain,
    validation_data=(xValidation , yValidation),
    epochs=20, # Increased epochs, as early stopping will handle when to stop
    batch_size=32,
    callbacks=[earlyStopping]
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(trainedModel.history['accuracy'])
plt.plot(trainedModel.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(trainedModel.history['loss'])
plt.plot(trainedModel.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
validationLoss, validationAccuracy = nnmodel.evaluate(xValidation,yValidation)

print("Validation Accuracy:" , validationAccuracy)

In [ ]:
testLoss, testAccuracy = nnmodel.evaluate(xTest,yTest)

print("Test Accuracy:", testAccuracy)

In [ ]:
yTestProbabilities = nnmodel.predict(xTest)

yTestPredictions = (yTestProbabilities > 0.5).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(yTest , yTestPredictions))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

confusionMatrix = confusion_matrix(yTest,yTestPredictions)

plt.figure(figsize=(6,5))

sns.heatmap( confusionMatrix,annot=True,fmt='d',cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Neural Network Confusion Matrix")

plt.show()

In [ ]:
train_accuracy = trainedModel.history['accuracy'][-1]
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {validationAccuracy:.4f}")
print(f"Test Accuracy: {testAccuracy:.4f}")


## Conclusion: Sentence Embedding + Neural Network Model Performance

After training and evaluating the Sentence Embedding-based Neural Network model for sentiment analysis on the IMDB movie review dataset, we can draw the following conclusions regarding its performance:

### 1. Performance Metrics

-   **Validation Accuracy (0.833):** The model achieved a validation accuracy of approximately 83.3%. This indicates its ability to generalize to unseen data during the training process.
-   **Test Accuracy (0.827):** The test accuracy is slightly lower than the validation accuracy at approximately 82.7%. This suggests that the model's performance on completely unseen data is consistent with its validation performance.
-   **Classification Report:** The precision, recall, and f1-score for both positive and negative classes are around 0.83. This indicates a balanced performance across classes.
-   **Confusion Matrix:** The confusion matrix shows `[[3042, 663], [622, 3111]]`. This means that out of 3705 actual negative reviews, 663 were predicted as positive (false positives), and out of 3733 actual positive reviews, 622 were predicted as negative (false negatives). This indicates a moderate level of misclassification.

### 2. Analysis and Potential Reasons for Performance:

-   **Model Complexity vs. Data Size/Task:** While sentence embeddings (like MiniLM used here) are powerful and capture rich contextual meaning, the subsequent simple neural network might not be complex enough to fully leverage all the information contained within these embeddings. A deeper or more sophisticated neural architecture, potentially with more layers or different activation functions, might be required to unlock the full potential.
-   **Hyperparameter Tuning:** The current neural network might not be optimally tuned. Aspects such as learning rate, number of layers, neurons per layer, dropout rates, and batch normalization parameters can significantly impact performance. Although `EarlyStopping` was used to prevent overfitting, a more exhaustive search of the hyperparameter space could lead to better results.
-   **Pre-trained Model Choice:** The `all-MiniLM-L6-v2` model is a good general-purpose embedding model. However, for this specific IMDB sentiment task, exploring other Sentence-BERT models or even fine-tuning a transformer model directly on sentiment data might yield improved performance.

# Overall Project Conclusion

This project aimed to perform sentiment analysis on the IMDB Movie Review dataset, classifying reviews as either positive or negative. We explored various machine learning approaches, from traditional methods using TF-IDF features to a neural network leveraging sentence embeddings. Below is a summary of the project's journey and key findings.

## 1. Reading the Data

*   **Data Loading:** The `IMDB Dataset.csv` containing 50,000 movie reviews and their sentiments was loaded into a pandas DataFrame.
*   **Initial Inspection:** Basic checks using `df.shape`, `df.info()`, and `df.isnull().sum()` confirmed the dataset's structure and absence of missing values.
*   **Duplicate Handling:** 418 duplicate reviews were identified and removed, reducing the dataset size from 50,000 to 49,582 unique entries. This ensures that the models are not biased by repeated samples.
*   **Class Distribution:** The sentiment distribution was found to be fairly balanced, with 24,884 positive and 24,698 negative reviews, which is ideal for binary classification tasks.

## 2. Text EDA (Exploratory Data Analysis) and Preprocessing

*   **Feature Engineering (Text Length):** New features `wordCount` and `charCount` were created to analyze review lengths. An initial exploration suggested that sentiment does not strongly correlate with review length.
*   **HTML Tag Removal:** Many reviews contained HTML tags (e.g., `<br /><br />`). These were effectively removed using `BeautifulSoup`, cleaning the text for further processing.
*   **Sentiment Encoding:** The categorical `sentiment` column ('positive', 'negative') was converted into a numerical format (1 for positive, 0 for negative) for model compatibility.
*   **Text Normalization:**
    *   All review text was converted to lowercase.
    *   URLs were identified and removed.
    *   Special characters, numbers, and punctuation were removed, leaving only alphabetic characters. Multiple spaces were consolidated, and leading/trailing spaces were trimmed.
*   **Stopword Removal:** Common English stopwords (e.g., 'the', 'is', 'a') were removed using NLTK's stopwords list. This step reduces noise and focuses on more meaningful words.
*   **Lemmatization:** Words were reduced to their base forms (e.g., 'movies' to 'movie', 'liked' to 'like') using NLTK's WordNetLemmatizer. This helps in reducing the vocabulary size and treating different word forms as a single feature.
*   **Word Frequency Analysis:** Analysis of the most frequent words revealed common terms in both overall reviews and sentiment-specific reviews, providing insights into the dataset's vocabulary.

## 3. Feature Extraction

*   **TF-IDF Vectorization:** The preprocessed text reviews were transformed into numerical features using `TfidfVectorizer`. We configured it to use `max_features=5000` and `ngram_range=(1, 2)` to capture both single words and two-word phrases, resulting in a TF-IDF matrix of shape (49582, 5000).
*   **Target Variable:** The `sentiment` column was set as the target variable `y`.
*   **Feature Importance:** The top 20 TF-IDF features were identified and visualized, highlighting words and phrases most indicative of sentiment.
*   **Data Splitting:** The data was split into training, validation, and test sets with a 70/15/15 ratio, ensuring a robust evaluation process. The shapes confirmed the splits: Training (34707, 5000), Validation (7437, 5000), and Testing (7438, 5000).
*   **Sparsity Check:** The TF-IDF matrix was confirmed to be sparse, as expected with this type of vectorization.

## 4. Using Machine Learning Models

### 4.1. Logistic Regression

*   **Model:** `LogisticRegression` was initialized with `max_iter=1000` and `random_state=42`.
*   **Performance:**
    *   **Validation Accuracy:** 0.884
    *   **Test Accuracy:** 0.887
    *   **Classification Report:** Showed high precision, recall, and f1-scores (around 0.88-0.90) for both classes, indicating excellent balanced performance.
    *   **Confusion Matrix:** Displayed very low numbers of false positives and false negatives, confirming its strong predictive capability.

### 4.2. Random Forest

*   **Model:** `RandomForestClassifier` was initialized with `n_estimators=100`, `random_state=42`, and `n_jobs=-1` for parallel processing.
*   **Performance:**
    *   **Validation Accuracy:** 0.846
    *   **Test Accuracy:** 0.844
    *   **Classification Report:** Achieved good precision, recall, and f1-scores (around 0.84-0.85) but slightly lower than Logistic Regression.
    *   **Confusion Matrix:** Indicated a higher number of misclassifications compared to Logistic Regression.

### 4.3. Neural Network with Sentence Embeddings

*   **Feature Extraction:** Text reviews were converted into dense numerical representations (embeddings) using `SentenceTransformer("all-MiniLM-L6-v2")`, resulting in a feature vector of 384 dimensions for each review.
*   **Data Splitting:** The embeddings were then split into training, validation, and test sets with the same 70/15/15 ratio.
*   **Model Architecture:** A `tf.keras.Sequential` neural network was constructed, consisting of:
    *   An input layer matching the embedding dimension (384).
    *   Two hidden `Dense` layers with `relu` activation (256 and 128 neurons respectively), each followed by `Dropout` (0.3) and `BatchNormalization` for regularization and stability.
    *   An output `Dense` layer with `sigmoid` activation for binary classification.
*   **Compilation:** The model was compiled with the `adam` optimizer, `binary_crossentropy` loss, and `accuracy` metric.
*   **Training:** The model was trained with `EarlyStopping` (patience=3) monitoring `val_loss` to prevent overfitting.
*   **Performance:**
    *   **Training Accuracy:** 0.8651 (at the last epoch before early stopping)
    *   **Validation Accuracy:** 0.833
    *   **Test Accuracy:** 0.827
    *   **Classification Report:** Showed balanced precision, recall, and f1-scores around 0.83 for both classes.
    *   **Confusion Matrix:** `[[3042, 663], [622, 3111]]` indicated moderate misclassification, with 663 false positives and 622 false negatives on the test set.

## 5. Overall Performance Summary and Recommendation

| Model                       | Validation Accuracy | Test Accuracy |
|:----------------------------|:--------------------|:--------------|
| Logistic Regression         | 0.884               | 0.887         |
| Random Forest               | 0.846               | 0.844         |
| Sentence Embedding + NN     | 0.833               | 0.827         |

Based on the comprehensive evaluation, the **Logistic Regression model stands out as the top performer** for this IMDB sentiment analysis task, achieving the highest test accuracy of 0.887. It demonstrates excellent generalization capabilities and balanced performance across all metrics.

While the Sentence Embedding + Neural Network model, leveraging powerful contextual embeddings, showed respectable performance, its accuracy (0.827) was lower than both Logistic Regression and Random Forest. This could be attributed to several factors:

*   **Model Complexity vs. Task:** For this specific binary sentiment classification task with TF-IDF features, simpler linear models like Logistic Regression proved highly effective, possibly due to the linear separability of the features capturing sentiment.
*   **Neural Network Architecture:** The relatively simple neural network architecture used might not have been complex enough to fully leverage the rich information within the sentence embeddings. More sophisticated or deeper neural networks, along with extensive hyperparameter tuning, might be necessary to unlock their full potential.
*   **Hyperparameter Tuning:** While `EarlyStopping` was implemented, a more thorough exploration of the neural network's hyperparameter space (e.g., learning rates, number of layers/neurons, dropout rates) could lead to significant improvements.

In conclusion, for this dataset and task, the **Logistic Regression model is the most effective and efficient choice**, offering superior accuracy with less computational overhead. The Sentence Embedding + Neural Network approach shows promise but requires further optimization and architectural refinement to surpass traditional methods in this context.